In [16]:
import requests
from bs4 import BeautifulSoup
import json
import csv
import pandas as pd

def fetch_crowdstrike_adversaries():
    url = "https://www.crowdstrike.com/adversaries/"
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    adversaries = set()
    for h3 in soup.find_all("h3"):
        text = h3.get_text(strip=True)
        if text and "Adversary" not in text:
            adversaries.add(text)
    return adversaries

def fetch_microsoft_actors():
    return {
        "Storm-0558", "Storm-0866", "Storm-1849", "Storm-2077",
        "Volt Typhoon", "Seashell Blizzard", "Star Blizzard", "Flax Typhoon"
    }

def load_static_actors():
    return {
        "Wicked Panda", "Refined Kitten", "Rocket Kitten", "Cozy Bear", "Fancy Bear", "Primitive Bear",
        "Scattered Spider", "Morphing Meerkat", "Mustang Panda", "Indrik Spider", "Mustard Tempest",
        "Cold River", "Static Kitten", "Charming Kitten", "Play Ransomware Group", "Lace Tempest",
        "ALPHV", "Flax Typhoon", "Venomous Bear", "Velvet Chollima", "Machete", "8220 Gang",
        "Kryptonite Panda", "Prophet Spider", "Affiliates", "Carbon Spider", "Strawberry Tempest",
        "noname(057)16", "UAT-4820", "LilacSquid", "LABRAT", "TA582", "Asylum Ambuscade", "Rhysida",
        "TAG-100", "Deep Panda", "Apothecary Spider", "Helix Kitten", "Unfurling Hemlock",
        "Crystalray", "Ghostwriter", "TA577", "Circuit Panda", "Anchor Panda", "Boriselchin",
        "TA578", "Dynamite Panda", "Hippo Team", "Revolver Rabbit", "Stardust Chollima", "XeGroup",
    }

def load_unc_groups():
    return {f"UNC{n}" for n in range(1000, 6000)}

def combine_and_save(actors, json_path, csv_path, py_path):
    sorted_actors = sorted(actors)

    # Save JSON
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"threat_actors": sorted_actors}, f, indent=2)

    # Save CSV
    with open(csv_path, "w", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ThreatActor"])
        for actor in sorted_actors:
            writer.writerow([actor])

    # Save Python file
    with open(py_path, "w", encoding="utf-8") as f:
        f.write("THREAT_ACTORS = [\n")
        for actor in sorted_actors:
            f.write(f'    "{actor}",\n')
        f.write("]\n")

if __name__ == "__main__":
    cs = fetch_crowdstrike_adversaries()
    ms = fetch_microsoft_actors()
    mitre = load_static_actors()
    unc = load_unc_groups()

    combined = cs.union(ms, mitre, unc)
    combine_and_save(
        actors=combined,
        json_path="threat_actors_full.json",
        csv_path="threat_actors_full.csv",
        py_path="threat_actors_full.py"
    )
    print(f"✅ Saved {len(combined)} threat actors to JSON, CSV, and PY files.")
    # Convert the combined set to a sorted list and then to a DataFrame
    combined_df = pd.DataFrame(sorted(combined), columns=["ThreatActor"])
    display(combined_df)


✅ Saved 5070 threat actors to JSON, CSV, and PY files.


,ThreatActor
0,8220 Gang
1,ALPHV
2,Adversaries potentially targeting you
3,Affiliates
4,Anchor Panda
...,...
5065,Venomous Bear
5066,Volt Typhoon
5067,Wicked Panda
5068,XeGroup


In [21]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from newspaper import Article, Config
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Load threat actor list from DataFrame
THREAT_ACTORS = combined_df["ThreatActor"].dropna().unique().tolist()

# Download stopwords
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# --- Threat Enrichment ---

def enrich_ip(ip, abuse_key=None, virustotal_key=None):
    info = {"ip": ip, "abuse_score": None, "virustotal_tags": [], "threatfox_iocs": []}

    # AbuseIPDB
    if abuse_key:
        try:
            resp = requests.get(
                "https://api.abuseipdb.com/api/v2/check",
                headers={"Key": abuse_key, "Accept": "application/json"},
                params={"ipAddress": ip, "maxAgeInDays": 90}
            )
            if resp.ok:
                result = resp.json()["data"]
                info["abuse_score"] = result.get("abuseConfidenceScore")
        except Exception as e:
            print(f"AbuseIPDB error for {ip}: {e}")

    # VirusTotal
    if virustotal_key:
        try:
            vt_url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
            headers = {"x-apikey": virustotal_key}
            resp = requests.get(vt_url, headers=headers)
            if resp.ok:
                data = resp.json()
                info["virustotal_tags"] = data["data"]["attributes"].get("tags", [])
        except Exception as e:
            print(f"VirusTotal error for {ip}: {e}")

    # ThreatFox
    try:
        payload = {"query": "search_ioc", "search_term": ip}
        resp = requests.post("https://threatfox.abuse.ch/api/v1/", json=payload)
        if resp.ok:
            entries = resp.json().get("data", [])
            info["threatfox_iocs"] = [e.get("threat_type") for e in entries if e.get("threat_type")]
    except Exception as e:
        print(f"ThreatFox error for {ip}: {e}")

    return info

# --- Scraper Helpers ---

def load_saved_links(file_path):
    saved_links = set()
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if not content:
                    return saved_links
                data = json.loads(content)
                for entry in data:
                    if 'link' in entry:
                        saved_links.add(entry['link'])
        except json.JSONDecodeError:
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump([], f)
    return saved_links

def fetch_search_results(url, headers, retries=3):
    for _ in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.exceptions.RequestException:
            time.sleep(2)
    return None

def extract_results(soup, limit=10):
    return soup.find_all('a', class_='title', limit=limit) if soup else []

def extract_article_with_newspaper(link, user_agent):
    try:
        config = Config()
        config.browser_user_agent = user_agent
        article = Article(link, config=config)
        article.download()
        article.parse()
        text = article.text
        summary = ' '.join(text.split()[:50])
        publish_date = article.publish_date
        publish_date_str = publish_date.strftime("%Y-%m-%d %H:%M:%S") if publish_date else "Unknown"
        return text, summary, publish_date_str
    except Exception:
        time.sleep(2)
        return None, None, None

def fallback_extraction(link, headers):
    try:
        response = requests.get(link, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        article_div = soup.find('div', class_='article-body') or soup.find('div', class_='content')
        text = article_div.get_text(separator=' ', strip=True) if article_div else soup.get_text(separator=' ', strip=True)
        summary = ' '.join(text.split()[:50])
        return text, summary, "Unknown"
    except Exception:
        return None, None, None

def extract_article(link, user_agent, headers):
    text, summary, publish_date = extract_article_with_newspaper(link, user_agent)
    if not text or len(text.split()) < 100:
        text, summary, publish_date = fallback_extraction(link, headers)
    return text, summary, publish_date

def extract_keywords(text, top_n=10):
    if not text:
        return []
    vectorizer = TfidfVectorizer(stop_words='english', max_features=50)
    tfidf_matrix = vectorizer.fit_transform([text.lower()])
    feature_array = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.toarray().flatten()
    sorted_indices = tfidf_scores.argsort()[::-1]
    return [feature_array[idx] for idx in sorted_indices[:top_n]]

def match_threat_actors(text, actor_list):
    return [actor for actor in actor_list if actor.lower() in text.lower()]

def save_to_json(data, file_path):
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    existing_data = json.load(f)
                except json.JSONDecodeError:
                    existing_data = []
        else:
            existing_data = []
        existing_data.append(data)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"Error saving to file: {e}")

# --- Main Workflow ---

def main():
    search_ips = ["102.165.16.161"]  # Replace with your actual IP list

    # Load API keys from env vars or set them here
    abuse_key = os.getenv("ABUSEIPDB_API_KEY")
    vt_key = os.getenv("VT_API_KEY")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }
    user_agent = headers['User-Agent']
    file_path = 'BingSearchData_IP_Threats.json'

    saved_links = load_saved_links(file_path)
    print(f"Loaded {len(saved_links)} saved links from previous runs.")

    desired_valid_articles = 10

    for ip in search_ips:
        print(f"\n🔍 Searching for: {ip}")

        # Enrich IP
        enrichment = enrich_ip(ip, abuse_key=abuse_key, virustotal_key=vt_key)
        print(f"📊 Enrichment for {ip}:\n{json.dumps(enrichment, indent=2)}")

        search_url = f"https://www.bing.com/news/search?q={ip}"
        soup = fetch_search_results(search_url, headers)
        candidate_links = extract_results(soup, limit=20)
        print(f"🔗 {len(candidate_links)} candidate links found for IP '{ip}'")

        valid_articles_count = 0
        seen_links = set()

        for result in candidate_links:
            link = result.get('href')
            if link in seen_links or link in saved_links:
                continue
            seen_links.add(link)

            title = result.get_text(strip=True)
            print(f"📰 Title: {title}")
            print(f"🔗 Link: {link}")
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

            content, summary, publish_date = extract_article(link, user_agent, headers)

            if content and len(content.split()) >= 100:
                found_actors = match_threat_actors(content, THREAT_ACTORS)
                keywords = extract_keywords(summary)

                data = {
                    'ip': ip,
                    'timestamp': timestamp,
                    'title': title,
                    'link': link,
                    'content': content,
                    'summary': summary,
                    'keywords': keywords,
                    'threat_actors': found_actors,
                    'publish_date': publish_date,
                    'enrichment': enrichment
                }

                try:
                    save_to_json(data, file_path)
                    saved_links.add(link)
                    valid_articles_count += 1
                except (TypeError, ValueError) as e:
                    print(f"❌ Error saving data for link {link}: {e}")

                print(f"✅ Valid articles so far: {valid_articles_count}/{desired_valid_articles}")

                if valid_articles_count >= desired_valid_articles:
                    break
            else:
                print("⚠️ Excluding this link due to insufficient content.")

        if valid_articles_count < desired_valid_articles:
            print(f"📉 {valid_articles_count} valid articles found for IP '{ip}'.")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jaskew\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loaded 0 saved links from previous runs.

🔍 Searching for: 102.165.16.161
ThreatFox error for 102.165.16.161: HTTPSConnectionPool(host='threatfox.abuse.ch', port=443): Max retries exceeded with url: /api/v1/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1002)')))
📊 Enrichment for 102.165.16.161:
{
  "ip": "102.165.16.161",
  "abuse_score": null,
  "virustotal_tags": [],
  "threatfox_iocs": []
}
🔗 0 candidate links found for IP '102.165.16.161'
📉 0 valid articles found for IP '102.165.16.161'.
